# Validação 04 — Identidade do artigo com Crossref

## Goal

Comprovar que um artigo recuperado no PubMed pode ter seu DOI resolvido no Crossref e seus metadados comparados para confirmar a identidade da publicação.

## Setup

Fontes: [E-utilities do NCBI](https://www.ncbi.nlm.nih.gov/books/NBK25499/) e [API REST oficial do Crossref](https://www.crossref.org/documentation/retrieve-metadata/rest-api/). O notebook consulta um PMID conhecido para manter a validação limitada e reproduzível. `CROSSREF_EMAIL` pode ser definido para utilizar o acesso identificado recomendado pelo Crossref.

In [1]:
from dataclasses import asdict
from datetime import datetime, timezone
from pathlib import Path
from pprint import pprint
import os
import sys

project_root = Path.cwd()
if not (project_root / "src").exists():
    project_root = project_root.parent

sys.path.insert(0, str(project_root / "src"))

from fatofake import (
    CrossrefClient,
    PubMedClient,
    prepare_search_plan,
    search_pubmed,
    validate_analysis_input,
    verify_publications,
)

executed_at = datetime.now(timezone.utc).isoformat()
print(f"Execução UTC: {executed_at}")

Execução UTC: 2026-09-23T13:28:02.105892+00:00


## Steps

Recuperamos o PMID `33431520` no PubMed, extraímos seu DOI e consultamos o registro correspondente no Crossref. A identidade é confirmada quando o DOI coincide e a similaridade dos títulos atinge o limite definido.

In [2]:
class PmidQueryPlanner:
    def generate_queries(self, claim: str) -> list[str]:
        return ["33431520[pmid]"]

In [3]:
analysis_input = validate_analysis_input(
    "O consumo de café altera o risco de câncer de próstata."
)
search_plan = prepare_search_plan(analysis_input, PmidQueryPlanner())
pubmed_client = PubMedClient(
    email=os.getenv("NCBI_EMAIL"),
    api_key=os.getenv("NCBI_API_KEY"),
)
pubmed_result = search_pubmed(search_plan, pubmed_client, max_results_per_query=1)
pprint([asdict(publication) for publication in pubmed_result.publications])

[{'authors': ('Chen X', 'Zhao Y', 'Tao Z', 'Wang K'),
  'doi': '10.1136/bmjopen-2020-038902',
  'journal': 'BMJ open',
  'matched_queries': ('33431520[pmid]',),
  'pmid': '33431520',
  'publication_date': '2021 Jan 11',
  'source': 'PubMed',
  'title': 'Coffee consumption and risk of prostate cancer: a systematic '
           'review and meta-analysis.',
  'url': 'https://pubmed.ncbi.nlm.nih.gov/33431520/'}]


In [4]:
crossref_client = CrossrefClient(
    email=os.getenv("CROSSREF_EMAIL") or os.getenv("NCBI_EMAIL"),
)
identity_results = verify_publications(
    pubmed_result.publications,
    crossref_client,
)
pprint([asdict(result) for result in identity_results])

[{'crossref_journal': 'BMJ Open',
  'crossref_publication_date': '2021-01',
  'crossref_publisher': 'BMJ',
  'crossref_title': 'Coffee consumption and risk of prostate cancer: a '
                    'systematic review and meta-analysis',
  'crossref_url': 'https://doi.org/10.1136/bmjopen-2020-038902',
  'doi': '10.1136/bmjopen-2020-038902',
  'pmid': '33431520',
  'pubmed_title': 'Coffee consumption and risk of prostate cancer: a '
                  'systematic review and meta-analysis.',
  'reason': 'DOI resolvido e título compatível entre PubMed e Crossref.',
  'status': 'VERIFIED',
  'title_similarity': 1.0}]


## Checks

As verificações confirmam que o mesmo DOI foi localizado e que os títulos do PubMed e do Crossref são compatíveis. Isso confirma a identidade bibliográfica, não a qualidade científica do artigo.

In [5]:
assert len(pubmed_result.publications) == 1
assert len(identity_results) == 1

publication = pubmed_result.publications[0]
identity = identity_results[0]

assert publication.pmid == "33431520"
assert publication.doi == "10.1136/bmjopen-2020-038902"
assert identity.status == "VERIFIED"
assert identity.title_similarity is not None
assert identity.title_similarity >= 0.85
assert identity.crossref_url

print(
    f"Identidade confirmada: PMID {identity.pmid}, DOI {identity.doi}, "
    f"similaridade de título {identity.title_similarity:.2%}."
)

Identidade confirmada: PMID 33431520, DOI 10.1136/bmjopen-2020-038902, similaridade de título 100.00%.


## Next Steps

A conferência de identidade estará validada quando todas as células forem executadas sem erros. A próxima etapa será obter o resumo e, quando disponível, o texto completo no PubMed Central para iniciar a análise das evidências.